# Module 1: Fine-tune Qwen3-0.6B bằng LoRA

## Tổng quan 3 bước:
1. **Sinh Silver Dataset**: Sử dụng Gemini API để trích xuất thông tin từ 50 file CV (raw text) thành định dạng JSON có cấu trúc.
2. **Fine-tune LoRA**: Sử dụng Unsloth để tải mô hình `Qwen3-0.6B` ở định dạng 4-bit, sau đó huấn luyện (SFT) với bộ dữ liệu silver dataset.
3. **Lưu & Kiểm tra**: Lưu mô hình LoRA Adapter và thực hiện kiểm tra nhanh để so sánh kết quả trích xuất của SLM với nhãn silver từ Gemini.
\n

In [2]:
# Cài đặt các thư viện cần thiết
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes
!pip install google-genai pydantic scikit-learn

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-udtg8ei9/unsloth_88696f5bc48946fab5b874a3f375d6f6
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-udtg8ei9/unsloth_88696f5bc48946fab5b874a3f375d6f6
  Resolved https://github.com/unslothai/unsloth.git to commit 154a5eaeb3c5694ab406e763653db8946f313a31
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 117.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 105.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 125.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 24.4 MB/s eta 0:00:00
   ━

In [5]:
import os, time, glob
OUTPUT_DIR = '/content/drive/MyDrive/cv-analysis-service/finetune_output/'

for p in sorted(glob.glob(os.path.join(OUTPUT_DIR, '**', '*.gguf'), recursive=True)):
    print(f"{p}\n   {os.path.getsize(p)/1e9:.2f} GB | {time.ctime(os.path.getmtime(p))}")

# Đọc kiến trúc và tên model ghi trong file (cần: pip install gguf)
!pip -q install gguf
from gguf import GGUFReader
for p in glob.glob(os.path.join(OUTPUT_DIR, '**', '*.gguf'), recursive=True):
    r = GGUFReader(p)
    for k in ("general.architecture", "general.name"):
        f = r.fields.get(k)
        if f is not None:
            print(os.path.basename(p), k, "=", bytes(f.parts[f.data[0]]).decode())

/content/drive/MyDrive/cv-analysis-service/finetune_output/gguf_gguf/Qwen3-1.7B.Q4_K_M.gguf
   1.11 GB | Thu Oct  8 03:04:40 2026
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.5/118.5 kB 6.8 MB/s eta 0:00:00
Qwen3-1.7B.Q4_K_M.gguf general.architecture = qwen3
Qwen3-1.7B.Q4_K_M.gguf general.name = Gguf


In [4]:
from google.colab import drive
drive.mount('/content/drive')

import glob, os, time
OUTPUT_DIR = '/content/drive/MyDrive/cv-analysis-service/finetune_output/'

print("Tồn tại:", os.path.exists(OUTPUT_DIR))
print("Nội dung:", os.listdir(OUTPUT_DIR))

for p in sorted(glob.glob(os.path.join(OUTPUT_DIR, '**', 'adapter_config.json'), recursive=True)):
    print(p, "|", time.ctime(os.path.getmtime(p)))

Mounted at /content/drive
Tồn tại: True
Nội dung: ['silver_dataset.jsonl', 'train.jsonl', 'val.jsonl', 'lora_adapter', 'merged_model', 'gguf', 'gguf_gguf']
/content/drive/MyDrive/cv-analysis-service/finetune_output/lora_adapter/adapter_config.json | Thu Oct  8 02:54:02 2026
/content/drive/MyDrive/cv-analysis-service/finetune_output/lora_adapter/checkpoint-200/adapter_config.json | Thu Oct  8 02:34:08 2026
/content/drive/MyDrive/cv-analysis-service/finetune_output/lora_adapter/checkpoint-250/adapter_config.json | Thu Oct  8 02:45:53 2026


In [6]:
from google.colab import drive
drive.mount('/content/drive')

from unsloth import FastLanguageModel   # phải import đầu tiên
import os, json, torch

assert torch.cuda.is_available(), "Chưa bật GPU: Runtime > Change runtime type > T4 GPU"

OUTPUT_DIR = '/content/drive/MyDrive/cv-analysis-service/finetune_output/'
adapter_path = os.path.join(OUTPUT_DIR, 'lora_adapter')
assert os.path.exists(os.path.join(adapter_path, 'adapter_config.json')), \
    "Không thấy adapter: kiểm tra Drive đã mount và đường dẫn"

# 1. Nạp adapter đã train (max_seq_length=8192 chỉ để suy luận, không ảnh hưởng adapter)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=adapter_path, max_seq_length=8192, dtype=None, load_in_4bit=True,
)
FastLanguageModel.for_inference(model)

# 2. Nạp tập val
with open(os.path.join(OUTPUT_DIR, 'val.jsonl'), encoding='utf-8') as f:
    val_records = [json.loads(l) for l in f]
print(f"Số mẫu val: {len(val_records)}")

# 3. Schema: GIỐNG HỆT cell 8 lúc train
JSON_SCHEMA_FORMAT = """
{
  "personal_info": {
    "full_name": null,
    "email": null,
    "phone": null,
    "github_url": null
  },
  "objective": null,
  "education": [
    {
      "school": null,
      "major": null,
      "start_year": null,
      "end_year": null,
      "gpa": null
    }
  ],
  "experience": [
    {
      "company": null,
      "job_title": null,
      "start_date": null,
      "end_date": null,
      "description": []
    }
  ],
  "personal_projects": [
    {
      "name": null,
      "role": null,
      "team_size": null,
      "start_date": null,
      "end_date": null,
      "github_url": null,
      "description": null,
      "tech_stack": []
    }
  ],
  "skills": [],
  "certificates": [],
  "honours": [],
  "languages": [
    {
      "language": null,
      "note": null
    }
  ]
}
"""

# 4. Test 3 mẫu đầu với prompt giống hệt lúc train
MAX_NEW = 2500
for s in val_records[:3]:
    prompt = (f"<|im_start|>system\n{s['instruction']}\n{JSON_SCHEMA_FORMAT}<|im_end|>\n"
              f"<|im_start|>user\n{s['input']}<|im_end|>\n<|im_start|>assistant\n")
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    n_in = inputs.input_ids.shape[1]
    out = model.generate(**inputs, max_new_tokens=MAX_NEW, do_sample=False, use_cache=True)
    gen = out[0][n_in:]
    pred = tokenizer.decode(gen, skip_special_tokens=True).strip()
    try:
        json.loads(pred); ok = True
    except Exception:
        ok = False
    print(f"\n===== {s['file']} | prompt={n_in} tok | output={len(gen)} tok "
          f"| json_ok={ok} | hit_cap={len(gen) >= MAX_NEW} "
          f"| chars: pred={len(pred)} vs nhãn={len(s['output'])} =====")
    print(pred[:600]); print("..."); print(pred[-300:])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
==((====))==  Unsloth 2026.10.2: Fast Qwen3 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Unsloth 2026.10.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.
Both `max_new_tokens` (=2500) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Số mẫu val: 99


Both `max_new_tokens` (=2500) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



===== 482.txt | prompt=1844 tok | output=2500 tok | json_ok=False | hit_cap=True | chars: pred=9759 vs nhãn=2440 =====
<think>
Okay, let's tackle this information extraction task. First, I need to parse the given CV carefully according to the provided guidelines. Let me start by reading through the entire CV and identifying each section.

**Personal Information:**
The CV starts with the personal details. The user provided:
- Full name: Nguyễn Thanh Bảo
- Email: nguyenthanhbao39767@outlook.com
- Phone: 081057438
- GitHub URL: https://github.com/nguyenthanhbao540
- Location: Remote (GMT+7)

I'll put this into the "personal_info" field. All these fields are present and should be copied directly.

**Objective:**
T
...
     "end_date": "Dec 2024",
      "tech_stack": [
        "TypeScript",
        "Node.js/NestJS",
        "Java/Spring Boot",
        "Elasticsearch"
      ],
      "description": "Cut page load time from 1200s to 320s through rendering optimizations. Service Booking System

Both `max_new_tokens` (=2500) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



===== 100.txt | prompt=2507 tok | output=2500 tok | json_ok=False | hit_cap=True | chars: pred=8704 vs nhãn=4162 =====
<think>
Okay, let's tackle this information extraction task. First, I need to parse the given CV carefully according to the provided schema. Let me start by reading through the entire CV to understand the structure and extract each section properly.

**Personal Information:**
The CV starts with Lê Đình Yến's details. The user provided full name, email, phone, GitHub URL. I'll assign these to the personal_info object.

**Objective:**
The objective section is a single line: "Mục tiêu là kỹ sư có 3 năm kinh nghiệm thiết kế và tối ưu hệ thống quản lý kho, mong muốn đảm nhiệm trách nhiệm về kiến tr
...
the user's CV, they have separate sections for each project. The original schema might not require personal_projects, so I can leave it as empty.

Similarly, the projects section in the schema is a separate field. So I should create a "projects" array if the CV has multiple 

In [7]:
import torch
lora = [(n, p) for n, p in model.named_parameters() if "lora_" in n]
print("Số tensor LoRA:", len(lora))
if lora:
    mx = max(p.detach().abs().max().item() for n, p in lora if "lora_B" in n)
    print("max|lora_B| =", mx)   # lora_B khởi tạo bằng 0, train xong phải > 0 rõ rệt

Số tensor LoRA: 392
max|lora_B| = 0.6488104462623596


In [8]:
import gc, os, json
assert 'val_records' in globals() and 'JSON_SCHEMA_FORMAT' in globals(), "Chạy lại cell test trước"

del model
gc.collect(); torch.cuda.empty_cache()

merged_path = os.path.join(OUTPUT_DIR, 'merged_model')
print(os.listdir(merged_path))

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=merged_path, max_seq_length=8192, dtype=None, load_in_4bit=True,
)
FastLanguageModel.for_inference(model)

MAX_NEW = 2500
for s in val_records[:2]:
    prompt = (f"<|im_start|>system\n{s['instruction']}\n{JSON_SCHEMA_FORMAT}<|im_end|>\n"
              f"<|im_start|>user\n{s['input']}<|im_end|>\n<|im_start|>assistant\n")
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    n_in = inputs.input_ids.shape[1]
    out = model.generate(**inputs, max_new_tokens=MAX_NEW, do_sample=False, use_cache=True)
    gen = out[0][n_in:]
    pred = tokenizer.decode(gen, skip_special_tokens=True).strip()
    try:
        json.loads(pred); ok = True
    except Exception:
        ok = False
    print(f"\n===== {s['file']} | output={len(gen)} tok | json_ok={ok} | hit_cap={len(gen) >= MAX_NEW} "
          f"| chars: pred={len(pred)} vs nhãn={len(s['output'])} =====")
    print(pred[:400]); print("..."); print(pred[-200:])

['chat_template.jinja', 'tokenizer_config.json', 'tokenizer.json', 'config.json', 'generation_config.json', '.cache', 'model.safetensors']
==((====))==  Unsloth 2026.10.2: Fast Qwen3 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

The tokenizer you are loading from '/content/drive/MyDrive/cv-analysis-service/finetune_output/merged_model' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
The tokenizer you are loading from '/content/drive/MyDrive/cv-analysis-service/finetune_output/merged_model' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
Both `max_new_tokens` (=2500) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/


===== 482.txt | output=1806 tok | json_ok=False | hit_cap=False | chars: pred=7225 vs nhãn=2440 =====
<think>
Okay, let's tackle this information extraction task. First, I need to parse the given CV carefully according to the guidelines. Let me start by breaking down each section.

**Personal Information:**
The CV starts with "Nguyễn Thanh Bảo" and some details. The user provided the full name, email, phone, and GitHub URL. So I'll extract those into the personal_info section. All these are scalar
...
oyee of the Month (Q3 2024)"
  ],
  "languages": [
    {
      "language": "Vietnamese",
      "note": "Native"
    },
    {
      "language": "English",
      "note": "Fluent - IELTS 7.5"
    }
  ]
}

===== 100.txt | output=2500 tok | json_ok=False | hit_cap=True | chars: pred=9042 vs nhãn=4162 =====
<think>
Okay, let's tackle this information extraction task. First, I need to parse the provided CV carefully according to the given guidelines. Let me start by understanding each section a

In [8]:
import os
from google.colab import drive
import time
import json
from google import genai
from pydantic import BaseModel
from typing import Optional, List

# Mount Google Drive
drive.mount('/content/drive')

# Khai báo đường dẫn
# INPUT_DIR = '/content/drive/MyDrive/cv-analysis-service/cv_texts/'
OUTPUT_DIR = '/content/drive/MyDrive/cv-analysis-service/finetune_output/'

os.makedirs(OUTPUT_DIR, exist_ok=True)

# # Kiểm tra thư mục input
# if not os.path.exists(INPUT_DIR) or len(os.listdir(INPUT_DIR)) == 0:
#     raise FileNotFoundError(f"Thư mục {INPUT_DIR} không tồn tại hoặc trống. Vui lòng kiểm tra lại Google Drive.")

# # Nhập API Key (Không hardcode)
# import getpass
# GEMINI_API_KEY = getpass.getpass('Nhập GEMINI_API_KEY: ')

# # Khởi tạo client Gemini
# client = genai.Client(api_key=GEMINI_API_KEY)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
# Định nghĩa JSON Schema (Pydantic)
class PersonalInfo(BaseModel):
    full_name: str
    email: Optional[str] = None
    phone: Optional[str] = None
    github_url: Optional[str] = None

class Education(BaseModel):
    school: str
    major: Optional[str] = None
    start_year: Optional[int] = None
    end_year: Optional[int] = None
    gpa: Optional[str] = None

class Experience(BaseModel):
    company: str
    job_title: Optional[str] = None
    start_date: Optional[str] = None
    end_date: Optional[str] = None
    description: List[str] = []

class Project(BaseModel):
    name: str
    role: Optional[str] = None
    team_size: Optional[int] = None
    start_date: Optional[str] = None
    end_date: Optional[str] = None
    github_url: Optional[str] = None
    description: Optional[str] = None
    tech_stack: List[str] = []

class LanguageSkill(BaseModel):
    language: str
    note: Optional[str] = None

class CandidateProfile(BaseModel):
    personal_info: PersonalInfo
    objective: Optional[str] = None
    education: List[Education] = []
    experience: List[Experience] = []
    personal_projects: List[Project] = []
    skills: List[str] = []
    certificates: List[str] = []
    honours: List[str] = []
    languages: List[LanguageSkill] = []

In [10]:
SYSTEM_PROMPT = """
Bạn là hệ thống trích xuất thông tin từ CV.

Nhiệm vụ:
Đọc DUY NHẤT nội dung CV được cung cấp và trả về JSON theo đúng schema.
Đây là bài toán INFORMATION EXTRACTION, không phải bài toán viết lại, tóm tắt
hay suy luận thông tin.

QUY TẮC BẮT BUỘC:

1. NGUỒN DỮ LIỆU
- Chỉ sử dụng thông tin xuất hiện trực tiếp trong CV được cung cấp.
- Không sử dụng kiến thức bên ngoài.
- Không sử dụng thông tin từ các CV khác, ví dụ khác hoặc câu trả lời trước đó.
- Không tự bịa, suy luận hoặc dự đoán thông tin còn thiếu.

2. THÔNG TIN KHÔNG CÓ
- Nếu một trường dạng scalar không xuất hiện trong CV → null.
- Nếu một trường dạng array không có dữ liệu → [].
- Không dùng 0, "", "N/A", "Unknown" để biểu diễn thông tin không có.

3. GIỮ NGUYÊN THÔNG TIN
- Giữ nguyên ngôn ngữ gốc của CV.
- Không dịch.
- Không tự ý sửa tên người, tên công ty, tên trường, tên project.
- Không tự ý chuẩn hóa hoặc thay đổi số điện thoại, URL, ngày tháng hoặc số liệu.
- Không thêm ngày/tháng/ngày cụ thể nếu CV không cung cấp.

4. URL
- Chỉ lấy URL xuất hiện trực tiếp trong CV.
- Giữ URL dưới dạng raw URL.
- Không chuyển URL thành Markdown.
- Không thêm text, dấu ngoặc hoặc formatting vào URL.
- Không tự tạo hoặc suy đoán URL.

5. NGÀY THÁNG
- Giữ nguyên mức độ chi tiết được cung cấp trong CV.
- Nếu CV chỉ có năm → chỉ ghi năm.
- Nếu CV có tháng và năm → giữ tháng và năm.
- Không tự thêm tháng hoặc ngày.
- Không suy ra ngày kết thúc từ ngày bắt đầu hoặc ngược lại.

6. ENTITY ATTRIBUTION
- Chỉ gán thông tin cho một entity khi CV thể hiện rõ thông tin đó thuộc entity tương ứng.
- Không suy diễn quan hệ giữa các thông tin chỉ vì chúng xuất hiện gần nhau.
- Không lấy thông tin của project này để điền vào project khác.
- Không lấy thông tin của experience này để điền vào experience khác.
- Không lấy thông tin ở header, objective hoặc profile để suy ra role/job_title của project hoặc experience.

7. SKILLS VÀ PROJECT TECH STACK
- "skills" là các kỹ năng/công nghệ được liệt kê trong phần Skills/Technical Skills chung của CV.
- "tech_stack" của project chỉ chứa công nghệ được CV gắn trực tiếp với project đó.
- Không copy toàn bộ skills chung vào tech_stack của project.
- Không copy tech_stack của project này sang project khác.
- Nếu một công nghệ chỉ xuất hiện trong Skills chung nhưng không được gắn với project → không đưa công nghệ đó vào project.
- Nếu một công nghệ xuất hiện trực tiếp trong project → có thể đưa vào tech_stack của project.

8. ROLE VÀ JOB TITLE
- Chỉ điền role/job_title nếu CV ghi rõ role/job title tương ứng.
- Không suy ra role từ objective, ngành học, skills hoặc tên công nghệ.
- Không suy ra role từ tên project.

9. TEAM SIZE
- Chỉ điền team_size khi CV ghi rõ số lượng thành viên của project/team.
- Không suy ra team size.
- Nếu không có → null.

10. DESCRIPTION
- Với experience.description: tách từng bullet/câu thành từng phần tử riêng biệt.
- Không gộp nhiều bullet thành một câu.
- Không bỏ mất thông tin định lượng hoặc tên công nghệ.
- Không tự tóm tắt hoặc paraphrase nội dung.
- Với project.description: giữ nội dung mô tả trung thực theo CV.

11. FRAMEWORK / TECHNOLOGY GROUP
- Nếu CV viết dạng nhóm như:
  "Spring Cloud - Gateway, Eureka, Config, Feign"
  thì khai triển thành:
  "Spring Cloud Gateway",
  "Spring Cloud Eureka",
  "Spring Cloud Config",
  "Spring Cloud Feign".
- Chỉ khai triển những module thực sự được CV liệt kê.
- Không tự thêm module không xuất hiện trong CV.

12. TÍNH ĐẦY ĐỦ
- Không bỏ qua thông tin có trong CV chỉ vì thông tin đó xuất hiện ở cuối CV.
- Đặc biệt phải kiểm tra đầy đủ các section:
  personal information,
  objective,
  education,
  experience,
  projects,
  skills,
  certificates,
  honours,
  languages.

13. OUTPUT
- Chỉ trả về JSON hợp lệ.
- Không trả về Markdown.
- Không trả về code fence.
- Không giải thích.
- Không thêm field ngoài schema.
- Không đổi tên key trong schema.
"""

In [11]:
# Chia tập train/val
from sklearn.model_selection import train_test_split
silver_dataset_path = os.path.join(OUTPUT_DIR, 'silver_dataset.jsonl')
with open(silver_dataset_path, 'r', encoding='utf-8') as f:
    records = [json.loads(line) for line in f]

train_records, val_records = train_test_split(records, test_size=0.2, random_state=42)

train_path = os.path.join(OUTPUT_DIR, 'train.jsonl')
val_path = os.path.join(OUTPUT_DIR, 'val.jsonl')

with open(train_path, 'w', encoding='utf-8') as f:
    for record in train_records:
        f.write(json.dumps(record, ensure_ascii=False) + '\n')

with open(val_path, 'w', encoding='utf-8') as f:
    for record in val_records:
        f.write(json.dumps(record, ensure_ascii=False) + '\n')

print(f"Đã chia {len(train_records)} mẫu train và {len(val_records)} mẫu val.")

Đã chia 395 mẫu train và 99 mẫu val.


In [12]:
from unsloth import FastLanguageModel
import torch

model_name = "unsloth/Qwen3-1.7B"
max_seq_length = 4096
dtype = None  # Tự động phát hiện
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.10.2: Fast Qwen3 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

In [13]:
# Gắn LoRA adapter
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
    use_rslora = False,
    loftq_config = None,
)

Unsloth 2026.10.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [14]:
# Format dataset theo chat template (Bơm thêm Schema vào Prompt để model không bịa key)
from datasets import Dataset

JSON_SCHEMA_FORMAT = """
{
  "personal_info": {
    "full_name": null,
    "email": null,
    "phone": null,
    "github_url": null
  },
  "objective": null,
  "education": [
    {
      "school": null,
      "major": null,
      "start_year": null,
      "end_year": null,
      "gpa": null
    }
  ],
  "experience": [
    {
      "company": null,
      "job_title": null,
      "start_date": null,
      "end_date": null,
      "description": []
    }
  ],
  "personal_projects": [
    {
      "name": null,
      "role": null,
      "team_size": null,
      "start_date": null,
      "end_date": null,
      "github_url": null,
      "description": null,
      "tech_stack": []
    }
  ],
  "skills": [],
  "certificates": [],
  "honours": [],
  "languages": [
    {
      "language": null,
      "note": null
    }
  ]
}
"""

chat_template = """<|im_start|>system
{instruction}
{schema}<|im_end|>
<|im_start|>user
{input}<|im_end|>
<|im_start|>assistant
{output}<|im_end|>
"""

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input_text, output in zip(instructions, inputs, outputs):
        text = chat_template.format(instruction=instruction, schema=JSON_SCHEMA_FORMAT, input=input_text, output=output)
        texts.append(text)
    return { "text" : texts, }

train_dataset = Dataset.from_json(train_path)
val_dataset = Dataset.from_json(val_path)

train_dataset = train_dataset.map(formatting_prompts_func, batched = True)
val_dataset = val_dataset.map(formatting_prompts_func, batched = True)


Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/395 [00:00<?, ? examples/s]

Map:   0%|          | 0/99 [00:00<?, ? examples/s]

In [15]:
from trl import SFTTrainer
from transformers import TrainingArguments

output_model_dir = os.path.join(OUTPUT_DIR, 'lora_adapter')

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=5,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1,
        eval_strategy="steps",
        eval_steps=10,
        save_strategy="epoch",
        save_total_limit=2,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir=output_model_dir,
        report_to="none"
    ),
)

trainer_stats = trainer.train()

print("\n--- ĐÁNH GIÁ KẾT QUẢ HUẤN LUYỆN ---")
history = trainer.state.log_history
train_losses = [log['loss'] for log in history if 'loss' in log]
eval_losses = [log['eval_loss'] for log in history if 'eval_loss' in log]

if train_losses:
    print(f"Train Loss cuối cùng: {train_losses[-1]:.4f}")
if eval_losses:
    print(f"Eval Loss cuối cùng: {eval_losses[-1]:.4f}")
    min_eval_loss = min(eval_losses)
    if eval_losses[-1] > min_eval_loss * 1.1:
        print("CẢNH BÁO: Có dấu hiệu Overfitting (Eval loss tăng trở lại).")

Unsloth: transformers renamed `push_to_hub_token` to `hub_token`. Forwarding your value to `hub_token` - update your code when convenient. If you also passed `hub_token` as None, that is its default here and cannot be distinguished from leaving it unset, so `push_to_hub_token` was used; drop `push_to_hub_token` to keep it.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/395 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/99 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 395 | Num Epochs = 5 | Total steps = 250
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 17,432,576 of 1,738,007,552 (1.00% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
10,1.671044,1.576772
20,0.612884,0.512537
30,0.033218,0.025106
40,0.008935,0.008442
50,0.003979,0.003365
60,0.000499,0.000473
70,0.000350,0.000341
80,0.000283,0.000277
90,0.000242,0.000239
100,0.000216,0.000213


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/cv-analysis-service/finetune_output/lora_adapter/checkpoint-50/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/cv-analysis-service/finetune_output/lora_adapter/checkpoint-100/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/cv-analysis-service/finetune_output/lora_adapter/checkpoint-150/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/cv-analysis-service/finetune_output/lora_adapter/checkpoint-200/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/cv-analysis-service/finetune_output/lora_adapter/checkpoint-250/tokenizer_config.json.



--- ĐÁNH GIÁ KẾT QUẢ HUẤN LUYỆN ---
Train Loss cuối cùng: 0.0001
Eval Loss cuối cùng: 0.0001


In [19]:
# 1. Adapter (để train tiếp / kiểm tra)
model.save_pretrained(output_model_dir)
tokenizer.save_pretrained(output_model_dir)
print(f"Đã lưu adapter vào: {output_model_dir}")

# 2. Merge full precision
MERGED_DIR = os.path.join(OUTPUT_DIR, "merged_model")
model.save_pretrained_merged(
    MERGED_DIR,
    tokenizer,
    save_method="merged_16bit",
)
print(f"Đã lưu merged model vào: {MERGED_DIR}")

# 3. GGUF Q4_K_M (file duy nhất cần tải về local để serve)
GGUF_DIR = os.path.join(OUTPUT_DIR, "gguf")
model.save_pretrained_gguf(
    GGUF_DIR,
    tokenizer,
    quantization_method="q4_k_m",
)
print(f"Đã lưu GGUF vào: {GGUF_DIR}")
print("Tải file .gguf về máy, đặt vào ai-service/merged_model/ rồi restart service.")

Đã lưu adapter vào: /content/drive/MyDrive/cv-analysis-service/finetune_output/lora_adapter


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/cv-analysis-service/finetune_output/merged_model/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.44GB            

...

model.safetensors: downloading bytes:           |  0.00B            

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [01:19<00:00, 79.68s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [01:49<00:00, 109.03s/it]


Unsloth: Merge process complete. Saved to `/content/drive/MyDrive/cv-analysis-service/finetune_output/gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b11443-mix-d65395f (app-b11443-mix-d65395f-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['/content/drive/MyDrive/cv-analysis-service/finetune_output/gguf_gguf/Qwen3-1.7B.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minut

In [14]:
import json as _json
import torch

FastLanguageModel.for_inference(model)

# Test 1 mẫu trước
N_EVAL = 10

n_json_ok = 0
n_exact = 0

for i in range(N_EVAL):
    sample = val_records[i]

    messages = [
        {
            "role": "system",
            "content": sample["instruction"] + f"""

{JSON_SCHEMA_FORMAT}

IMPORTANT:
- Return ONLY one valid JSON object.
- Do NOT provide reasoning.
- Do NOT provide analysis.
- Do NOT output <think>...</think>.
- Do NOT output Markdown.
- Do NOT output code fences.
- Do NOT explain the answer.
- Do NOT wrap URLs in Markdown links.
- Do NOT wrap email addresses in Markdown links.
- github_url MUST be a raw URL such as https://github.com/username.
- email MUST be a raw email address such as example@gmail.com.
- Do NOT add fields that are not present in the schema.
- Do NOT invent information that is not explicitly present in the CV.
"""
        },
        {
            "role": "user",
            "content": sample["input"]
        }
    ]

    # Qwen3 chat template + tắt thinking
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    # Generate
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=8192,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            return_dict_in_generate=True,
        )

    # Chỉ lấy token model sinh ra
    gen = out.sequences[0][inputs["input_ids"].shape[1]:]

    # Kiểm tra model có dừng bằng EOS không
    last_token = gen[-1].item() if len(gen) > 0 else None
    eos_token = tokenizer.eos_token_id

    print("=" * 80)
    print(f"File: {sample['file']}")
    print(f"Generated tokens: {len(gen)}")
    print(f"Last token: {last_token}")
    print(f"EOS token: {eos_token}")
    print(f"Stopped by EOS: {last_token == eos_token}")

    # Decode
    pred = tokenizer.decode(
        gen,
        skip_special_tokens=True
    ).strip()

    print(f"Generated chars: {len(pred)}")

    # Validate JSON
    try:
      parsed = _json.loads(pred)
      ok = isinstance(parsed, dict)
      error_msg = None
    except Exception as e:
      ok = False
      error_msg = str(e)

    # So sánh exact với silver label
    exact = pred.strip() == sample["output"].strip()

    n_json_ok += int(ok)
    n_exact += int(exact)

    print(f"JSON valid: {ok}")
    print(f"Exact match: {exact}")

    if not ok:
        print("\nValidation error:")
        print(error_msg[:1000])

    print("\nMODEL OUTPUT:")
    print(pred)

    print("\nSILVER OUTPUT:")
    print(sample["output"])

print("=" * 80)
print(
    f"JSON hợp lệ: {n_json_ok}/{N_EVAL} | "
    f"Khớp toàn bộ silver: {n_exact}/{N_EVAL}"
)

Both `max_new_tokens` (=8192) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8192) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


File: 482.txt
Generated tokens: 925
Last token: 151645
EOS token: 151645
Stopped by EOS: True
Generated chars: 3234
JSON valid: True
Exact match: False

MODEL OUTPUT:
{
  "personal_info": {
    "full_name": "Nguyễn Thanh Bảo",
    "email": "nguyenthanhbao39767@outlook.com",
    "phone": "081057438",
    "github_url": "https://github.com/nguyenthanhbao540"
  },
  "objective": "Software engineer with 2 years of experience building work management system systems, seeking a Full-stack Developer role with broader ownership.",
  "education": [
    {
      "school": "Bách Khoa Hà Nội",
      "major": "Computer Science",
      "start_year": 2019,
      "end_year": 2022,
      "gpa": "3.24/4.0"
    }
  ],
  "experience": [
    {
      "company": "Viettel PAY",
      "job_title": "Full-stack Developer",
      "start_date": "Jul 2025",
      "end_date": "Present",
      "description": [
        "Used Git to sustain 5000 concurrent requests in production.",
        "Developed and maintained hệ thố

Both `max_new_tokens` (=8192) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


File: 100.txt
Generated tokens: 1399
Last token: 151645
EOS token: 151645
Stopped by EOS: True
Generated chars: 4301
JSON valid: True
Exact match: False

MODEL OUTPUT:
{
  "personal_info": {
    "full_name": "Lê Đình Yến",
    "email": "leinhyen1573@hotmail.com",
    "phone": "081670721",
    "github_url": "https://github.com/leinhyen881"
  },
  "objective": "Kỹ sư có 3 năm kinh nghiệm thiết kế và tối ưu hệ thống hệ thống quản lý kho, mong muốn đảm nhiệm trách nhiệm về kiến trúc hệ thống và chất lượng kỹ thuật.",
  "education": [
    {
      "school": "Trường Đại học Quốc gia TP.HCM",
      "major": "Khoa học Máy tính",
      "start_year": "2022",
      "end_year": "2025",
      "gpa": "3.31/4.0"
    }
  ],
  "experience": [
    {
      "company": "Shopee Vietnam",
      "job_title": "Backend Developer",
      "start_date": "01/2025",
      "end_date": "Hiện tại",
      "description": [
        "Xây dựng pipeline CI/CD tự động, rút ngắn thời gian deploy từ 2500 phút xuống 600 phút.",
 

Both `max_new_tokens` (=8192) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

File: 182.txt
Generated tokens: 1228
Last token: 151645
EOS token: 151645
Stopped by EOS: True
Generated chars: 4625
JSON valid: True
Exact match: False

MODEL OUTPUT:
{
  "personal_info": {
    "full_name": "Bạch Anh Toàn",
    "email": "bachanhtoan9766@gmail.com",
    "phone": "+84 48 9202189",
    "github_url": "https://github.com/bachanhtoan823"
  },
  "objective": "Senior engineer with 8 years designing distributed systems and leading engineers, seeking a Senior Frontend Engineer role at a large-scale product company.",
  "education": [
    {
      "school": "FPT",
      "major": "Software Engineering",
      "start_year": "2022",
      "end_year": "2025",
      "gpa": "8.5/10.0"
    }
  ],
  "experience": [
    {
      "company": "Traveloka",
      "job_title": "Senior Frontend Engineer",
      "start_date": "Jan 2025",
      "end_date": "Present",
      "description": [
        "Served as principal architect for 18 services, balancing cost, latency and availability trade-offs.",

Both `max_new_tokens` (=8192) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


File: 68.txt
Generated tokens: 989
Last token: 151645
EOS token: 151645
Stopped by EOS: True
Generated chars: 3913
JSON valid: True
Exact match: False

MODEL OUTPUT:
{
  "personal_info": {
    "full_name": "NGUYEN PHAM NHAT ANH",
    "email": "gmail.com",
    "phone": "thu Duc, Ho Chi Minh City",
    "github_url": "github.com/nhatanh-dev"
  },
  "objective": "Software Engineering student at FPT University with hands-on experience in ASP.NET Core, RESTful APIs, relational databases, and AWS-based deployment through team projects. Seeking a Backend Developer Internship to contribute to backend development, testing, and system integration.",
  "education": [
    {
      "school": "FPT University",
      "major": "Software Engineering",
      "start_year": "2023",
      "end_year": "2027",
      "gpa": "8.6"
    }
  ],
  "experience": [
    {
      "company": "EV Station Management System",
      "job_title": "Backend Developer Intern",
      "start_date": "09/2025",
      "end_date": "11/

Both `max_new_tokens` (=8192) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


File: 347.txt
Generated tokens: 1274
Last token: 151645
EOS token: 151645
Stopped by EOS: True
Generated chars: 4036
JSON valid: True
Exact match: False

MODEL OUTPUT:
{
  "personal_info": {
    "full_name": "Thái Khắc Quang",
    "email": "thaikhacquang26212@hotmail.com",
    "phone": "+84 98 4874182",
    "github_url": "https://github.com/thaikhacquang523"
  },
  "objective": "Kỹ sư có 3 năm kinh nghiệm thiết kế và tối ưu hệ thống quy trình hoàn tiền, mong muốn đảm nhiệm trách nhiệm về kiến trúc hệ thống và chất lượng kỹ thuật.",
  "education": [
    {
      "school": " Trường Đại học FPT ",
      "major": " Kỹ thuật Phần mềm ",
      "start_year": " 2020 ",
      "end_year": " 2023 ",
      "gpa": " 3.94/4.0 "
    }
  ],
  "experience": [
    {
      "company": " Simbicon ",
      "job_title": " Senior Business Analyst ",
      "start_date": " 04/2025 ",
      "end_date": " Hiện tại ",
      "description": [
        "Thiết kế chiến lược cache và phân tán tải, giảm tải cơ sở dữ liệu 